# Bài thực hành số 1 - Lý thuyết

**Xây dựng và triển khai một hợp đồng thông minh cơ bản bằng Vyper**

---
## Mục tiêu

- Trình bày được nguyên lý hoạt động của blockchain và smart contract.
- Hiểu và giải thích các khái niệm cơ bản trong Vyper.
- Cài đặt và thiết lập môi trường phát triển Vyper.
- Viết và triển khai một hợp đồng thông minh cơ bản.
- Kiểm tra và xác minh chức năng của hợp đồng trên testnet.

---
# Setup môi trường (Windows)

## 1. Kiểm tra Python

Nếu máy đã cài Python rồi, kiểm tra phiên bản (cần **3.10+**):

In [1]:
!python --version

Python 3.13.13


Hoặc chạy các cell bên dưới.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python    :", sys.version)
print("Executable:", sys.executable)
print("Platform  :", platform.platform())
print("CWD       :", Path.cwd())

assert platform.system() == "Windows", "Notebook nay chi danh cho Windows."
major, minor = sys.version_info[:2]
assert (major, minor) >= (3, 10), f"Can Python >= 3.10, hien tai {major}.{minor}"
print()
print("[OK] Python / Windows")

## 2. Tạo / kích hoạt virtualenv (PowerShell)

Mở **PowerShell** tại thư mục gốc project:

```powershell
cd d:\Study\IS355
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

Nếu PowerShell báo lỗi *execution policy*:

```powershell
Set-ExecutionPolicy -Scope CurrentUser RemoteSigned
```

Khi activate thành công, prompt có tiền tố `(.venv)`. Sau đó chọn kernel **Python (IS355 .venv)** (mục 5).

In [1]:
import sys
from pathlib import Path

prefix = Path(sys.prefix)
in_venv = ('.venv' in prefix.as_posix()) or (sys.prefix != sys.base_prefix)
print('sys.prefix :', sys.prefix)
print('Dang dung venv?', 'CO [OK]' if in_venv else 'KHONG - chay Activate.ps1 roi chon lai kernel')

candidates = [
    Path.cwd() / '.venv',
    Path.cwd().parent / '.venv',
    Path.cwd().parent.parent / '.venv',
]
found = next((p for p in candidates if (p / 'Scripts' / 'python.exe').exists()), None)
if found:
    print('Tim thay .venv :', found)
    print('Python trong venv:', found / 'Scripts' / 'python.exe')
else:
    print('Chua co .venv - chay lenh PowerShell o muc 2.')

sys.prefix : d:\Study\IS355\.venv
Dang dung venv? CO [OK]
Tim thay .venv : d:\Study\IS355\.venv
Python trong venv: d:\Study\IS355\.venv\Scripts\python.exe


## 3. Cài thư viện (download vyper + cryptography)

| Package | Mục đích |
|---------|----------|
| `cryptography` | Chữ ký số RSA (mục B) |
| `vyper` | Biên dịch hợp đồng `.vy` trên máy |
| `jupyter` / `ipykernel` | Chạy notebook |

PowerShell (đã Activate `.venv`):

```powershell
python -m pip install -U pip
python -m pip install cryptography vyper jupyter ipykernel
```

Có thể dùng thêm [Remix](https://remix.ethereum.org) (plugin Vyper) để Deploy trên trình duyệt.

In [2]:
%pip install -U pip
%pip install cryptography vyper jupyter ipykernel

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.5/1.8 MB 480.1 kB/s eta 0:00:03
   ----------- ---------------------------- 0.5/1.8 MB 480.1 kB/s eta 0:00:03
   ----------- ---------------------------- 0.5/1.8 MB 480.1 kB/s eta 0:00:03
   ----------------- ---------------------- 0.8/1.8 MB 418.9 kB/s eta 0:00:03
   ----------------- ---------------------- 0.8/1.8 MB 418.9 kB/s eta 0:00:03
   ----------------- ---------------------- 0.8/1.8 MB 418.9 kB/s eta 0:00:03
   ----------------------- ---------------- 1.0/1

In [3]:
import importlib.metadata as md

packages = ['cryptography', 'vyper', 'jupyter', 'ipykernel']
print(f"{'Package':<16} Version")
print('-' * 28)
for name in packages:
    try:
        print(f'{name:<16} {md.version(name)}')
    except md.PackageNotFoundError:
        print(f'{name:<16} CHUA CAI')

Package          Version
----------------------------
cryptography     50.0.1
vyper            0.4.3
jupyter          1.1.1
ipykernel        7.3.0


## 4. (Tuỳ chọn) OpenSSL trên Windows

Không bắt buộc - notebook dùng `cryptography` tự sinh khóa.

```powershell
choco install openssl
openssl genrsa -out ty.pem 2048
openssl rsa -in ty.pem -pubout -out ty.pub
```

Hoặc tải file `.exe`: https://github.com/openssl/openssl/wiki/Binaries

In [ ]:
import shutil
import subprocess

openssl = shutil.which('openssl')
if openssl:
    out = subprocess.check_output([openssl, 'version'], text=True).strip()
    print('[OK] OpenSSL:', out)
    print('  Path   :', openssl)
else:
    print('[SKIP] OpenSSL chua co trong PATH (dung cryptography cung duoc).')

## 5. Đăng ký kernel Jupyter cho `.venv`

In [4]:
import sys
import subprocess

cmd = [
    sys.executable,
    '-m',
    'ipykernel',
    'install',
    '--user',
    '--name=is355-venv',
    '--display-name=Python (IS355 .venv)',
]
print('Running:', ' '.join(cmd))
subprocess.check_call(cmd)
print()
print('[OK] Kernel da dang ky: Python (IS355 .venv)')
print('  Cursor: chon kernel nay o goc tren ben phai notebook.')

Running: d:\Study\IS355\.venv\Scripts\python.exe -m ipykernel install --user --name=is355-venv --display-name=Python (IS355 .venv)

[OK] Kernel da dang ky: Python (IS355 .venv)
  Cursor: chon kernel nay o goc tren ben phai notebook.


## 6. Smoke test

Kiểm tra: SHA-256 (blockchain), chữ ký số (`cryptography`), biên dịch Vyper.

In [5]:
import hashlib
import json

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa

# --- Blockchain hash ---
payload = json.dumps({'id': 1, 'history': 'Ty likes cat'}, sort_keys=True).encode()
digest = hashlib.sha256(payload).hexdigest()
print('SHA-256 sample:', digest[:32], '...')

# --- Digital signature ---
private_key = rsa.generate_private_key(65537, 2048, default_backend())
message = b'Ty likes cat'
signature = private_key.sign(
    message,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)
private_key.public_key().verify(
    signature,
    message,
    padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
    hashes.SHA256(),
)
print('[OK] cryptography (sign/verify)')

# --- Vyper compile ---
import vyper
from vyper.compiler import compile_code

sample = (
    '# @version ^0.4.0\n'
    '\n'
    'number: uint256\n'
    '\n'
    '@external\n'
    'def store(num: uint256):\n'
    '    self.number = num\n'
    '\n'
    '@external\n'
    'def retrieve() -> uint256:\n'
    '    return self.number\n'
)
result = compile_code(sample)
bytecode = result.get('bytecode') or ''
print('Vyper version :', vyper.__version__)
print('Bytecode (dau):', str(bytecode)[:48], '...')
print('[OK] vyper compile')
print('[OK] Setup xong - keo xuong phan bai tap ly thuyet ben duoi.')

SHA-256 sample: 1edbb2e508758f92a2e435fa20226128 ...
[OK] cryptography (sign/verify)
Vyper version : 0.4.3
Bytecode (dau): 0x61004361000f6000396100436000f35f3560e01c636057 ...
[OK] vyper compile
[OK] Setup xong - keo xuong phan bai tap ly thuyet ben duoi.


---
# Phần bài tập lý thuyết (tại lớp)

## A. Blockchain

Blockchain là một **cơ sở dữ liệu chỉ có thể thêm vào** (*append-only database*), gồm các **khối (blocks)** được liên kết với nhau bằng **hàm băm (hashing)**.

Mỗi khối chứa nhiều giao dịch chuyển giá trị (hoặc thông tin khác) giữa những người tham gia, được bảo mật bằng **mật mã học (cryptography)**. Một sự **đồng thuận (consensus)** giữa nhiều **nút (nodes)** nắm giữ cùng một cơ sở dữ liệu sẽ quyết định khối mới nào được thêm tiếp theo.

### Đặc tính nổi bật

- Không thể thay đổi dữ liệu đã ghi.
- Minh bạch và có thể kiểm chứng.
- Được bảo mật nhờ mật mã học và cơ chế đồng thuận.

### Bitcoin vs Ethereum

| | Bitcoin | Ethereum |
|---|---------|----------|
| Nội dung khối | Chủ yếu lưu giao dịch (vd: Tý gửi 1 BTC cho Tèo) | Giao dịch + **thay đổi trạng thái (change of state)** |
| Ví dụ trạng thái | — | Hàng đợi mua vé: trống hoặc đầy |

Ngoài danh sách giao dịch, khối còn lưu: thời gian thêm khối, độ khó mục tiêu (*target difficulty*), và **hash của khối cha (*parent hash*)**.

### Chuỗi khối (chain)

Mỗi khối lưu ID/hash của khối cha → tạo thành chuỗi liên kết:

```
Block A (ID=1, Parent=-)  →  Block B (ID=2, Parent=1)  →  Block C (ID=3, Parent=2)
  Tx1, Tx2, Tx3                 Tx4, Tx5                     Tx6, Tx7
```

### Thực hành: tạo chuỗi khối đơn giản bằng Python

Mỗi khối lưu `parent_id` và `parent_hash` (SHA-256 của khối cha) để tạo liên kết bất biến.

In [6]:
import hashlib
import json


class Block:
    id = None
    history = None
    parent_id = None
    parent_hash = None


def block_hash(block: Block) -> str:
    """Tính SHA-256 của toàn bộ thuộc tính khối."""
    payload = json.dumps(block.__dict__, sort_keys=True).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


block_A = Block()
block_A.id = 1
block_A.history = "Ty likes cat"

block_B = Block()
block_B.id = 2
block_B.history = "Teo likes dog"
block_B.parent_id = block_A.id
block_B.parent_hash = block_hash(block_A)

block_C = Block()
block_C.id = 3
block_C.history = "Mai likes dog"
block_C.parent_id = block_B.id
block_C.parent_hash = block_hash(block_B)

chain = [block_A, block_B, block_C]
for b in chain:
    print(f"Block {b.id}")
    print(f"  history     : {b.history}")
    print(f"  parent_id   : {b.parent_id}")
    print(f"  parent_hash : {b.parent_hash}")
    print(f"  self_hash   : {block_hash(b)}")
    print()

Block 1
  history     : Ty likes cat
  parent_id   : None
  parent_hash : None
  self_hash   : 1edbb2e508758f92a2e435fa20226128dac7499ca3f14b9c1d949329e24dc5b7

Block 2
  history     : Teo likes dog
  parent_id   : 1
  parent_hash : 1edbb2e508758f92a2e435fa20226128dac7499ca3f14b9c1d949329e24dc5b7
  self_hash   : 46bed02f0ca83b09c4df73e2e7fefe66b8331a0bfdceba8e94709b3c6329b4db

Block 3
  history     : Mai likes dog
  parent_id   : 2
  parent_hash : 46bed02f0ca83b09c4df73e2e7fefe66b8331a0bfdceba8e94709b3c6329b4db
  self_hash   : 811aebc35da972c4693f5595b49ce6c302933aa4f798a4512f88dbf1733d8323



### Thử nghiệm: sửa dữ liệu khối cha → hash không còn khớp

Nếu thay đổi `history` của Block A, `parent_hash` trong Block B sẽ **không còn đúng** — minh họa tính bất biến của blockchain.

In [7]:
# Giả sử kẻ tấn công sửa Block A
block_A.history = "Ty hates cat"

expected = block_B.parent_hash
actual = block_hash(block_A)

print("parent_hash lưu trong Block B:", expected)
print("hash Block A sau khi bị sửa  :", actual)
print("Chuỗi còn hợp lệ?", expected == actual)

parent_hash lưu trong Block B: 1edbb2e508758f92a2e435fa20226128dac7499ca3f14b9c1d949329e24dc5b7
hash Block A sau khi bị sửa  : 19a6fedcc944890d7ecf80470f3e93b5bc2f8e0c6ba5d17883f106b56e3efd9d
Chuỗi còn hợp lệ? False


---
## B. Chữ ký số lên dữ liệu trong blockchain

Trong blockchain dùng **hai khóa** để ký số dữ liệu: xác thực thông điệp và bảo vệ khỏi bị sửa bởi người trái phép.

| Khóa | Vai trò |
|------|---------|
| **Private key** (khóa riêng) | Giữ bí mật; dùng để **ký** thông điệp |
| **Public key** (khóa công khai) | Công khai; dùng để **xác minh** chữ ký |

### Tạo khóa bằng OpenSSL (tuỳ chọn, Windows)

Cài OpenSSL bằng Chocolatey, hoặc tải `.exe` tại https://github.com/openssl/openssl/wiki/Binaries

```powershell
choco install openssl
openssl genrsa -out ty.pem 2048
openssl rsa -in ty.pem -pubout -out ty.pub
```

> Trong notebook bên dưới, ta dùng thư viện `cryptography` của Python để **tự sinh khóa** (không bắt buộc OpenSSL).

### Cài thư viện

In [8]:
# Chạy 1 lần nếu chưa có cryptography
%pip install cryptography -q

Note: you may need to restart the kernel to use updated packages.


### TySignature — cấu hình, ký và xác minh

Tương đương file `TySignature.py` trong bài giảng.

In [9]:
from pathlib import Path

from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding, rsa

# === Configuration ===
GENERATE_PRIVATE_KEY = True
DERIVE_PUBLIC_KEY_FROM_PRIVATE_KEY = True
PRIVATE_KEY_FILE = "ty.pem"
PUBLIC_KEY_FILE = "ty.pub"
MESSAGE = b"Ty likes cat"

# --- Private key ---
if GENERATE_PRIVATE_KEY:
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
        backend=default_backend(),
    )
    Path(PRIVATE_KEY_FILE).write_bytes(
        private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.TraditionalOpenSSL,
            encryption_algorithm=serialization.NoEncryption(),
        )
    )
    print(f"Đã tạo private key → {PRIVATE_KEY_FILE}")
else:
    private_key = serialization.load_pem_private_key(
        Path(PRIVATE_KEY_FILE).read_bytes(),
        password=None,
        backend=default_backend(),
    )
    print(f"Đã nạp private key từ {PRIVATE_KEY_FILE}")

# --- Sign ---
signature = private_key.sign(
    MESSAGE,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH,
    ),
    hashes.SHA256(),
)

print("MESSAGE :", MESSAGE)
print("SIGNATURE (hex):", signature.hex()[:64], "...")
print("SIGNATURE length:", len(signature), "bytes")

Đã tạo private key → ty.pem
MESSAGE : b'Ty likes cat'
SIGNATURE (hex): 1683f62dc5884a81d809c023e923b606e2794697d04e69594da15e99c12cc1f4 ...
SIGNATURE length: 256 bytes


**Kết quả:** file khóa `ty.pem` / `ty.pub` đã tạo trong thư mục bài.

![File explorer](../Images/hinh_ty_pem_explorer.png)

![Nội dung ty.pem](../Images/hinh_ty_pem_noidung.png)

![Nội dung ty.pub](../Images/hinh_ty_pub_noidung.png)


In [10]:
from pathlib import Path

# --- Public key ---
if DERIVE_PUBLIC_KEY_FROM_PRIVATE_KEY:
    public_key = private_key.public_key()
    Path(PUBLIC_KEY_FILE).write_bytes(
        public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo,
        )
    )
    print(f"Đã derive + lưu public key → {PUBLIC_KEY_FILE}")
else:
    public_key = serialization.load_pem_public_key(
        Path(PUBLIC_KEY_FILE).read_bytes(),
        backend=default_backend(),
    )
    print(f"Đã nạp public key từ {PUBLIC_KEY_FILE}")

# --- Verify ---
try:
    public_key.verify(
        signature,
        MESSAGE,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )
    print("✓ Xác minh thành công: chữ ký hợp lệ với MESSAGE gốc.")
except Exception as e:
    print("✗ Xác minh thất bại:", e)

Đã derive + lưu public key → ty.pub
✓ Xác minh thành công: chữ ký hợp lệ với MESSAGE gốc.


### Yêu cầu thực hành tại lớp (mục B)

1. **Vai Tèo:** cố sửa thông điệp của Tý thành `"Ty hates cat"` rồi kiểm tra chữ ký.
2. Chạy lại với thông điệp gốc `"Ty likes cat"`, chụp chữ ký; đổi thành `"Ty hates cat"`, chạy lại và **so sánh** hai chữ ký.
3. *(*)* Viết chương trình xác thực xem có phải Tý đã viết `"Ty likes cat"` hay không.

In [11]:
# Yêu cầu 1 & 2: Tèo sửa thông điệp → chữ ký không còn hợp lệ
TAMPERED = b"Ty hates cat"

print("Chữ ký gốc (64 hex đầu):", signature.hex()[:64])

# Ký lại với thông điệp đã sửa (nếu Tý ký lại) → chữ ký khác hoàn toàn
sig_tampered = private_key.sign(
    TAMPERED,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH,
    ),
    hashes.SHA256(),
)
print("Chữ ký mới (Ty hates cat):", sig_tampered.hex()[:64])
print("Hai chữ ký giống nhau?", signature == sig_tampered)

# Tèo giữ chữ ký gốc nhưng đổi nội dung → verify FAIL
try:
    public_key.verify(
        signature,
        TAMPERED,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH,
        ),
        hashes.SHA256(),
    )
    print("Verify với thông điệp đã sửa: PASS (không mong đợi!)")
except Exception:
    print("Verify với thông điệp đã sửa: FAIL ← đúng như kỳ vọng (toàn vẹn bị phá)")

Chữ ký gốc (64 hex đầu): 1683f62dc5884a81d809c023e923b606e2794697d04e69594da15e99c12cc1f4
Chữ ký mới (Ty hates cat): 1a497afd600b5e541d103510f2dc33a7b2add80fc4d2544d1b12932e46c35fdb
Hai chữ ký giống nhau? False
Verify với thông điệp đã sửa: FAIL ← đúng như kỳ vọng (toàn vẹn bị phá)


In [12]:
# Yêu cầu 3 (*): xác thực Tý có phải người viết "Ty likes cat" không?

def verify_ty_message(message: bytes, sig: bytes, pub_key) -> bool:
    try:
        pub_key.verify(
            sig,
            message,
            padding.PSS(
                mgf=padding.MGF1(hashes.SHA256()),
                salt_length=padding.PSS.MAX_LENGTH,
            ),
            hashes.SHA256(),
        )
        return True
    except Exception:
        return False


claimed = b"Ty likes cat"
ok = verify_ty_message(claimed, signature, public_key)
print(f"Thông điệp: {claimed!r}")
print("Có phải Tý (chủ private key) đã ký?", "CÓ ✓" if ok else "KHÔNG ✗")

Thông điệp: b'Ty likes cat'
Có phải Tý (chủ private key) đã ký? CÓ ✓


---
## C. Smart contract

**Smart contract (hợp đồng thông minh)** là chương trình sống trên blockchain, giải quyết hạn chế của Bitcoin (chỉ giao dịch tài chính).

| | Bitcoin | Ethereum / Smart contract |
|---|---------|---------------------------|
| Giao dịch | Chuyển giá trị (vd: Tý gửi 1 BTC cho Tèo) | **Thay đổi trạng thái chương trình** (vd: biến từ 5 → 9) |
| Bản chất | Tài chính | State transition của chương trình |

### Cách hiểu trực quan

- Smart contract ≈ file nhị phân (như `.exe` trên Windows) nhưng **sống trên blockchain**.
- Chương trình tồn tại dưới dạng **bytecode dựa trên stack**.
- Khi tương tác, người dùng gửi lệnh → **EVM (Ethereum Virtual Machine)** đẩy/kéo giá trị từ stack và thực thi.
- Khác ứng dụng web: bytecode smart contract **minh bạch**, mọi nút đều giữ bản sao giống nhau.

> **Hình 2.** Smart contract tồn tại trên mọi node trong chuỗi khối.  
> **Hình 3.** Người dùng gửi input (instruction) tới smart contract.

---
## D. Sử dụng Vyper hiện thực một hợp đồng thông minh

### i. Môi trường triển khai trên Ethereum (Remix)

1. Mở trình duyệt → [https://remix.ethereum.org](https://remix.ethereum.org)
2. Trong **FILE EXPLORER**, mở thư mục `contracts`.
3. Click phải file `1_Storage.sol` → **Compile**.
4. Vào **Deploy & run transactions** → Deploy lên môi trường Remix VM / testnet.
5. Kiểm tra kết quả triển khai (địa chỉ hợp đồng, các hàm `store` / `retrieve`).

### ii. Thiết lập Vyper

**Trên Remix (web):**
1. Mở **Plugin Manager** → tìm `vyper` → Activate.
2. Xuất hiện icon compiler Vyper trên sidebar.
3. Chọn file `.vy` → Compile.

### iii. Hợp đồng đơn giản đầu tiên — `Storage.vy`

Biên dịch / Deploy trên **Remix** (plugin Vyper) — không cần cài Vyper local.

**Kết quả Remix:** biên dịch và triển khai `1_Storage.sol` trên Remix VM.

![Remix IDE - trang chủ](../Images/hinh_remix_home.png)

![Compile 1_Storage.sol](../Images/hinh_remix_compile.png)

![Deploy & Run Transactions](../Images/hinh_remix_deploy.png)

![Deployed Contracts](../Images/hinh_remix_deployed.png)

![Transactions history](../Images/hinh_remix_tx_history.png)


**Kết quả thiết lập Vyper trên Remix:**

![Plugin Manager trên sidebar](../Images/hinh_remix_plugin_sidebar.png)

![Vyper Compiler plugin Active](../Images/hinh_remix_vyper_plugin.png)


#### Mã nguồn `Storage.vy` (Hình 10)

Sau mỗi cell ghi file `.vy`, chạy **cell ngay bên dưới** để biên dịch và in kết quả (ABI + bytecode).

In [13]:
from pathlib import Path

# Nội dung file Storage.vy — chạy cell này để ghi file ra thư mục hiện tại
storage_vy = '''# @version ^0.4.3

number: uint256

@external
def store(num: uint256):
    self.number = num

@external
def retrieve() -> uint256:
    return self.number
'''

Path("Storage.vy").write_text(storage_vy, encoding="utf-8")
print("Da ghi Storage.vy")


Đã ghi Storage.vy
# @version ^0.4.3

number: uint256

@external
def store(num: uint256):
    self.number = num

@external
def retrieve() -> uint256:
    return self.number



In [21]:
# Ket qua bien dich: Storage.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("Storage.vy").read_text(encoding="utf-8")
print("File:", Path("Storage.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\Storage.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: store, retrieve
Bytecode (64 ky tu dau): 0x61004361000f6000396100436000f35f3560e01c636057361d811861002057 ...
Bytecode length: 270 chars


---
## E. Ngôn ngữ lập trình Vyper

### i. Kiểu dữ liệu

| Nhóm | Ví dụ |
|------|--------|
| Số nguyên có dấu | `int8`, `int64`, `int128`, `int256` |
| Số nguyên không dấu | `uint8`, `uint64`, `uint128`, `uint256` |
| Boolean | `bool` |
| Địa chỉ | `address` |
| Mảng kích thước cố định | `int16[5]` |
| Mảng Byte | `bytes32`, `Bytes[56]` |
| Chuỗi | `String[100]` |
| Enum / Flag | `flag Direction: ...` |
| List / DynArray | `DynArray[int128, 5]` |
| Struct | `struct Permission: ...` |
| Mapping | `HashMap[address, uint256]` |

#### `DataType.vy` (Hình 13) — khai báo các kiểu

In [14]:
from pathlib import Path

datatype_vy = '''# @version ^0.4.3

life_is_beautiful: bool

var_int1: int8
var_int2: int64
var_int3: int128
var_int4: int256

var_uint1: uint8
var_uint2: uint64
var_uint3: uint128
var_uint4: uint256

my_grandma_wallet: address

var_byte1: bytes32
var_byte2: bytes18
var_bytes: Bytes[56]

author: String[100]

flag Direction:
    NORTH
    SOUTH
    WEST
    EAST

direction: Direction

my_list: int16[5]
my_dynamic_array: DynArray[int128, 5]

struct Permission:
    write: bool
    execute: bool

my_permission: Permission
donaturs: HashMap[address, uint256]
'''

Path("DataType.vy").write_text(datatype_vy, encoding="utf-8")
print("Da ghi DataType.vy")


Đã ghi DataType.vy


In [22]:
# Ket qua bien dich: DataType.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("DataType.vy").read_text(encoding="utf-8")
print("File:", Path("DataType.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\DataType.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: (khong co)
Bytecode (64 ky tu dau): 0x61000361000f6000396100036000f35f5ffd8558208b8a5669fa721aa836a7 ...
Bytecode length: 140 chars


### ii. Hàm (Functions)

Hàm trong Vyper giống Python: nhóm câu lệnh trong một khối và thực thi khi được gọi.

- Ví dụ đã thấy: `store` / `retrieve` trong `Storage.vy`.
- Giống Python, nếu có `__init__` thì hàm này chạy **khi deploy** (constructor).
- Trong Vyper 0.4.x, constructor dùng decorator `@deploy`.

#### `StorageInit.vy` (Hình 14) — hàm khởi tạo

In [15]:
from pathlib import Path

storage_init_vy = '''# @version ^0.4.3

my_grandma_wallet: address
author: String[100]
fib_list: int16[5]
fib_dynamic_array: DynArray[int128, 5]

struct Permission:
    write: bool
    execute: bool

my_permission: Permission
donaturs: HashMap[address, uint256]

@deploy
def __init__():
    self.my_grandma_wallet = 0xde93510CFa39Ab92BF927399F799DbE71997Ee0b
    self.author = "Arjuna Sky Kok"
    self.fib_list = [1, 1, 2, 3, 5]
    self.fib_dynamic_array.append(1)
    self.fib_dynamic_array.append(1)
    self.fib_dynamic_array.append(2)
    self.my_permission = Permission(write=True, execute=False)
    self.donaturs[self.my_grandma_wallet] = 4000000000000000000
    donation_target: int128 = 8000000000000000000
'''

Path("StorageInit.vy").write_text(storage_init_vy, encoding="utf-8")
print("Da ghi StorageInit.vy")


Đã ghi StorageInit.vy


In [23]:
# Ket qua bien dich: StorageInit.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("StorageInit.vy").read_text(encoding="utf-8")
print("File:", Path("StorageInit.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\StorageInit.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: (khong co)
Bytecode (64 ky tu dau): 0x346100f75773de93510cfa39ab92bf927399f799dbe71997ee0b5f55600e60 ...
Bytecode length: 612 chars


### iii. Mutability (tính biến đổi)

Decorator tầm vực: `@external`, `@internal`.

Decorator **mutability**:

| Decorator | Đọc state / env? | Ghi state? | Nhận ETH? |
|-----------|------------------|------------|-----------|
| `@pure` | Không | Không | Không |
| `@view` | Có | Không | Không |
| `@nonpayable` | Có | Có | Không |
| `@payable` | Có | Có | **Có** |

Smart contract có thể hoạt động như **escrow / ngân hàng** nhờ khả năng nhận và giữ ETH.

#### `Annotations.vy` (Hình 15)

In [16]:
from pathlib import Path

annotations_vy = '''# @version ^0.4.3

author: String[100]
donatur: String[100]

@deploy
def __init__():
    # da ngam dinh @external
    self.author = "Arjuna Sky Kok"

@external
@pure
def add(x: int128, y: int128) -> int128:
    return x + y

@external
@view
def get_name_and_title() -> String[200]:
    return concat("Mr. ", self.author)

@external
@nonpayable
def change_name(new_name: String[100]):
    self.author = new_name

@external
@payable
def donate(donatur_name: String[100]):
    self.donatur = donatur_name
'''

Path("Annotations.vy").write_text(annotations_vy, encoding="utf-8")
print("Da ghi Annotations.vy")


Đã ghi Annotations.vy
→ Biên dịch & Deploy trên Remix, ghi kết quả vào báo cáo.


In [24]:
# Ket qua bien dich: Annotations.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("Annotations.vy").read_text(encoding="utf-8")
print("File:", Path("Annotations.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\Annotations.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: add, get_name_and_title, change_name, donate
Bytecode (64 ky tu dau): 0x3461004e57600e6040527f41726a756e6120536b79204b6f6b000000000000 ...
Bytecode length: 1386 chars


### iv. Cấu trúc điều khiển

Vyper hỗ trợ `if` / `elif` / `else` và vòng `for` giống Python.

In [17]:
from pathlib import Path

# ControlFlow.vy (Hình 16)
control_vy = '''# @version ^0.4.3

@external
@pure
def sum() -> uint256:
    s: uint256 = 0
    for i: uint256 in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
        s += i
    return s

@external
@pure
def greet(time: String[10]) -> String[20]:
    if time == "morning":
        return "Good morning!"
    elif time == "evening":
        return "Good evening!"
    else:
        return "How are you?"
'''

Path("ControlFlow.vy").write_text(control_vy, encoding="utf-8")
print("Da ghi ControlFlow.vy")


Đã ghi ControlFlow.vy
→ Biên dịch & Deploy trên Remix, ghi kết quả vào báo cáo.


In [25]:
# Ket qua bien dich: ControlFlow.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("ControlFlow.vy").read_text(encoding="utf-8")
print("File:", Path("ControlFlow.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\ControlFlow.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: sum, greet
Bytecode (64 ky tu dau): 0x61027e6100116100003961027e610000f35f3560e01c63853255cc81186100 ...
Bytecode length: 1418 chars


### v. Biến môi trường

| Biến | Ý nghĩa |
|------|---------|
| `block.number` | Số thứ tự khối hiện tại |
| `block.timestamp` | Timestamp của khối |
| `msg.sender` | Địa chỉ người gọi |
| `msg.value` | Số ETH (wei) gửi kèm giao dịch |

In [18]:
from pathlib import Path

# EnvVar.vy (Hình 17)
envvar_vy = '''# @version ^0.4.3

donatur: address
donation: uint256
time: uint256

@external
@payable
def donate():
    self.donatur = msg.sender
    self.donation = msg.value
    self.time = block.timestamp
'''

Path("EnvVar.vy").write_text(envvar_vy, encoding="utf-8")
print("Da ghi EnvVar.vy")


Đã ghi EnvVar.vy
→ Biên dịch & Deploy trên Remix, ghi kết quả vào báo cáo.


In [26]:
# Ket qua bien dich: EnvVar.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("EnvVar.vy").read_text(encoding="utf-8")
print("File:", Path("EnvVar.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\EnvVar.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: donate
Bytecode (64 ky tu dau): 0x61002061000f6000396100206000f35f3560e01c63ed88c68e811861001c57 ...
Bytecode length: 200 chars


### vi. Event logging

Ngoài biến trạng thái, có thể **ghi nhật ký sự kiện** (`log`) để theo dõi / thông báo off-chain (vd: đạt mục tiêu quyên góp → đăng mạng xã hội).

In [19]:
from pathlib import Path

# EventLogging.vy (Hình 18)
event_vy = '''# @version ^0.4.3

event Donation:
    donatur: indexed(address)
    amount: uint256

@external
@payable
def donate():
    log Donation(msg.sender, msg.value)
'''

Path("EventLogging.vy").write_text(event_vy, encoding="utf-8")
print("Da ghi EventLogging.vy")


Đã ghi EventLogging.vy
→ Biên dịch & Deploy trên Remix, ghi kết quả vào báo cáo.


In [27]:
# Ket qua bien dich: EventLogging.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("EventLogging.vy").read_text(encoding="utf-8")
print("File:", Path("EventLogging.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\EventLogging.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: donate
Bytecode (64 ky tu dau): 0x61004061000f6000396100406000f35f3560e01c63ed88c68e811861003c57 ...
Bytecode length: 264 chars
D:\Study\IS355\.venv\Lib\site-packages\vyper\semantics\types\user.py:313: Deprecation: Instantiating events with positional arguments is deprecated as of v0.4.1 and will be disallowed in a future release. Use kwargs instead e.g.:
```
log Donation(donatur=msg.sender, amount=msg.value)
```

  function "donate", line 10:8 
        9 def donate():
  ---> 10     log Donation(msg.sender, msg.value)
  ----------------^
       11

  vyper_warn(Deprecation(msg, node))


### vii. Interface

Smart contract có thể gọi smart contract khác qua **interface**.

Ví dụ: `StorageClient.vy` gọi hàm `retrieve()` của `Storage.vy` đã deploy trước đó.

In [20]:
from pathlib import Path

# StorageClient.vy (Hình 19)
client_vy = '''# @version ^0.4.3

interface Storage:
    def retrieve() -> uint256: view

storage_contract: Storage

@deploy
def __init__(storage_address: address):
    self.storage_contract = Storage(storage_address)

@external
def call_retrieve() -> uint256:
    return staticcall self.storage_contract.retrieve()
'''

Path("StorageClient.vy").write_text(client_vy, encoding="utf-8")
print("Da ghi StorageClient.vy")


Đã ghi StorageClient.vy
→ Deploy Storage.vy trước, lấy địa chỉ, rồi Deploy StorageClient với địa chỉ đó.


In [28]:
# Ket qua bien dich: StorageClient.vy
from pathlib import Path
from vyper.compiler import compile_code

_src = Path("StorageClient.vy").read_text(encoding="utf-8")
print("File:", Path("StorageClient.vy").resolve())
print("-" * 50)
try:
    result = compile_code(_src, output_formats=["abi", "bytecode"])
    abi = result.get("abi") or []
    fns = [x.get("name") for x in abi if isinstance(x, dict) and x.get("type") == "function"]
    bytecode = str(result.get("bytecode") or "")
    print("[OK] Bien dich thanh cong")
    print("ABI functions:", ", ".join(fns) if fns else "(khong co)")
    print("Bytecode (64 ky tu dau):", bytecode[:64], ("..." if len(bytecode) > 64 else ""))
    print("Bytecode length:", len(bytecode), "chars")
except Exception as e:
    print("[FAIL]", type(e).__name__)
    print(e)


File: D:\Study\IS355\TH\w1\BTTH_TrenLop\StorageClient.vy
--------------------------------------------------
[OK] Bien dich thanh cong
ABI functions: call_retrieve
Bytecode (64 ky tu dau): 0x3461002d5760206100b25f395f518060a01c61002d576040526040515f5561 ...
Bytecode length: 358 chars


**Kết quả:** thư mục `BTTH_TrenLop` sau khi chạy notebook (các file `.vy`, `ty.pem`, `ty.pub`).

![BTTH_TrenLop explorer](../Images/hinh_btth_trenlop_explorer.png)
